# Atividade Prática 5


Aluno: João Gabriel Angelo Bradachi

Professor: Fabrício Silva

Objetivo: Aplicar o método de PCA para reduzir a dimensão do conjunto de dados breast-cancer-wisconsin.csv


In [7]:
# imports
import pandas as pd

from time import time

import optuna
import xgboost as xgb

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [2]:
# Obtendo dados
df_cancer = pd.read_csv("breastcancerwisconsin.csv")
df_cancer

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,NaN
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,NaN
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,NaN
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,NaN


## Análise exploratória

Vamos fazer uma pequena análise exploratória e ajustar os dados para o PCA


In [3]:
df_cancer.describe()

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
count,5.690000e+02,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,0.0
mean,3.037183e+07,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,NaN
std,1.250206e+08,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,NaN
min,8.670000e+03,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,NaN
25%,8.692180e+05,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,NaN
50%,9.060240e+05,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,NaN
75%,8.813129e+06,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,NaN
max,9.113205e+08,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,NaN


In [4]:
df_cancer.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

## Testando no XGB sem PCA

Vamos treinar com tudo, fazer um fine tunning dos hiper-parametros com cross validation

In [ ]:
# Treinamento com cross validation

# Adicionando o label
encoder = LabelEncoder()

X = df_cancer.drop(columns=['id', 'diagnosis', 'Unnamed: 32'])
y = encoder.fit_transform(df_cancer['diagnosis'])

def objective(trial):
  params = {
      'n_estimators': trial.suggest_int('n_estimators', 50, 300),
      'max_depth': trial.suggest_int('max_depth', 3, 9),
      'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
      'subsample': trial.suggest_float('subsample', 0.5, 1.0),
      'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
      'random_state': 42,
  }
  # params = {
  #       'n_estimators': 200,
  #       'max_depth': 8,
  #       'learning_rate': 0.25,
  #       'subsample': 0.72,
  #       'colsample_bytree': 0.79,
  #       'random_state': 42,
  #   }

  model = xgb.XGBClassifier(**params)

  # Validação cruzada com 5 folds
  scores = cross_val_score(model, X, y, cv=5, scoring='accuracy', verbose=1)
  return scores.mean()


# Executando o estudo
start = time()
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=20)
print(f"Tempo para treinar 5 vezes com 5 cross validations: {time()-start} segundos")
print('Melhores parâmetros:', study.best_params)

[I 2026-09-22 21:08:38,647] A new study created in memory with name: no-name-a16c41be-0a98-411c-9d70-5a6194f41f9b
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.6s finished
[I 2026-09-22 21:08:39,228] Trial 0 finished with value: 0.9753920198726906 and parameters: {}. Best is trial 0 with value: 0.9753920198726906.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.5s finished
[I 2026-09-22 21:08:39,764] Trial 1 finished with value: 0.9753920198726906 and parameters: {}. Best is trial 0 with value: 0.9753920198726906.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.5s finished
[I 2026-09-22 21:08:40,289] Trial 2 finished with value: 0.9753920198726906 and parameters: {}. Best is trial 0 with value: 0.9753920198726906.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.6s finished
[I 2026-09-22 21:08:40,922] Trial 3 finished with value: 0.9753920198726906 and parameters: {}. Best is trial 0 with value: 0.9753920198726906.
[Parallel(n_jobs=1)]: Done   5

Tempo para treinar 5 vezes com 5 cross validations: 11.062309503555298 segundos
Melhores parâmetros: {}


## Resultados


In [19]:
melhores_parametros = study.best_params

X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_pipeline = Pipeline(steps=[
    ('classificador', XGBClassifier(**melhores_parametros))
])

print("Treinando o modelo...")
modelo_pipeline.fit(X_treino, y_treino)

print("Fazendo previsões no conjunto de teste...")
previsoes = modelo_pipeline.predict(X_teste)
categorias_texto = encoder.inverse_transform(previsoes)

print("\nRelatório de Classificação:")
print(classification_report(y_teste, previsoes, digits=5))

Treinando o modelo...
Fazendo previsões no conjunto de teste...

Relatório de Classificação:
              precision    recall  f1-score   support

           0    0.95890   0.98592   0.97222        71
           1    0.97561   0.93023   0.95238        43

    accuracy                        0.96491       114
   macro avg    0.96726   0.95807   0.96230       114
weighted avg    0.96521   0.96491   0.96474       114



Sem o PCA a acurácia e a precisão ficaram entre 0.974+-0.002 com o tempo de treino de 11.062 segundos


## Testando no XGB com PCA

In [16]:
# O PCA é sensível a escala
# Necessário normalizar

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA com 95% da variância original
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Colunas originais: {X.shape[1]}")
print(f"Colunas após PCA: {X_pca.shape[1]}")
print(f"Variância retida: {sum(pca.explained_variance_ratio_) * 100:.2f}%")

Colunas originais: 30
Colunas após PCA: 10
Variância retida: 95.16%


In [23]:
def objective(trial):
  params = {
        'n_estimators': 200,
        'max_depth': 8,
        'learning_rate': 0.25,
        'subsample': 0.72,
        'colsample_bytree': 0.79,
        'random_state': 42,
    }
  # params = {
  #     'n_estimators': trial.suggest_int('n_estimators', 50, 300),
  #     'max_depth': trial.suggest_int('max_depth', 3, 9),
  #     'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
  #     'subsample': trial.suggest_float('subsample', 0.5, 1.0),
  #     'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
  #     'random_state': 42,
  # }

  model = xgb.XGBClassifier(**params)

  # Validação cruzada com 5 folds
  scores = cross_val_score(model, X_pca, y, cv=5, scoring='accuracy', verbose=1)
  return scores.mean()


# Executando o estudo
start = time()
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=20)
print(f"Tempo para treinar 5 vezes com 5 cross validations: {time()-start} segundos")
print('Melhores parâmetros:', study.best_params)

[I 2026-09-22 21:09:40,006] A new study created in memory with name: no-name-006811dd-553e-49bc-9989-5e2aa7fb16de
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.6s finished
[I 2026-09-22 21:09:40,619] Trial 0 finished with value: 0.9701288619779538 and parameters: {}. Best is trial 0 with value: 0.9701288619779538.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.3s finished
[I 2026-09-22 21:09:40,911] Trial 1 finished with value: 0.9701288619779538 and parameters: {}. Best is trial 0 with value: 0.9701288619779538.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.6s finished
[I 2026-09-22 21:09:41,528] Trial 2 finished with value: 0.9701288619779538 and parameters: {}. Best is trial 0 with value: 0.9701288619779538.
[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    0.5s finished
[I 2026-09-22 21:09:42,027] Trial 3 finished with value: 0.9701288619779538 and parameters: {}. Best is trial 0 with value: 0.9701288619779538.
[Parallel(n_jobs=1)]: Done   5

Tempo para treinar 5 vezes com 5 cross validations: 8.01838755607605 segundos
Melhores parâmetros: {}


In [18]:
melhores_parametros = study.best_params

X_treino, X_teste, y_treino, y_teste = train_test_split(X_pca, y, test_size=0.2, random_state=42)

modelo_pipeline = Pipeline(steps=[
    ('classificador', XGBClassifier(**melhores_parametros))
])

print("Treinando o modelo...")
modelo_pipeline.fit(X_treino, y_treino)

print("Fazendo previsões no conjunto de teste...")
previsoes = modelo_pipeline.predict(X_teste)
categorias_texto = encoder.inverse_transform(previsoes)

print("\nRelatório de Classificação:")
print(classification_report(y_teste, previsoes, digits=5))

Treinando o modelo...
Fazendo previsões no conjunto de teste...

Relatório de Classificação:
              precision    recall  f1-score   support

           0    0.98592   0.98592   0.98592        71
           1    0.97674   0.97674   0.97674        43

    accuracy                        0.98246       114
   macro avg    0.98133   0.98133   0.98133       114
weighted avg    0.98246   0.98246   0.98246       114



## Resultados

Com o uso do PCA o tempo de treino foi menor, 8.018 segundos, e as métricas foram melhores, variando em 0.981+-0.004


## Conclusão

Nesse trabalho foi possível observar e colocar em prática o uso do PCA. A prática foi feita com sucesso, e pude perceber que, reduzindo as colunas, o tempo de treinamento diminui e a acurácia se mantém (ou até mesmo aumenta)